# Reward Uplift Lab on the real Criteo Uplift dataset

The simulated study validates every method against known ground truth. This notebook runs the parts that are observable on real data: the sample-ratio check, the average effect, and the comparison of uplift models by validation and test Qini area.

The dataset (about 14 million users, 12 anonymised features, ~85% treated) is published by Criteo AI Lab: Diemert et al., *A Large Scale Benchmark for Uplift Modeling*, AdKDD 2018. Download `criteo-research-uplift-v2.1.csv.gz` from the [Criteo AI Lab dataset page](https://ailab.criteo.com/criteo-uplift-prediction-dataset/) and check its licence before use.

In [ ]:
!pip install -q git+https://github.com/manojrevenkar12-web/reward-uplift-lab.git

Download the file (about 300 MB compressed). The cell checks the result and stops with a clear message if the download failed. If the direct link has moved, copy the current link from the [dataset page](https://ailab.criteo.com/criteo-uplift-prediction-dataset/) into `URL`, or upload the file to Colab yourself and put it at `data/criteo-research-uplift-v2.1.csv.gz`.

In [ ]:
import gzip, os, subprocess

# The original go.criteo.net link no longer serves the file; this is a mirror of the same
# file published with the libuplift package (311 MB, identical columns).
URL = 'https://github.com/jszymon/uplift_sklearn_data/releases/download/Criteo/criteo-research-uplift-v2.1.csv.gz'
PATH = 'data/criteo-research-uplift-v2.1.csv.gz'
os.makedirs('data', exist_ok=True)

if not os.path.exists(PATH) or os.path.getsize(PATH) < 100_000_000:
    subprocess.run(['wget', '--progress=dot:giga', '-O', PATH, URL], check=False)

size_mb = os.path.getsize(PATH) / 1e6 if os.path.exists(PATH) else 0
print(f'{PATH}: {size_mb:,.0f} MB')
if size_mb < 100:
    raise RuntimeError('Download failed or is incomplete. Get the current link from the dataset page, '
                       'or upload the file to data/ and re-run this cell.')
with gzip.open(PATH, 'rt') as fh:
    print('header:', fh.readline().strip())

A 10% sample (about 1.4 million users) keeps the run to a few minutes on a free Colab CPU. Sampling users uniformly preserves the randomisation.

In [ ]:
result = subprocess.run(
    ['uplift-lab', '-v', 'criteo', PATH, '--outcome', 'visit', '--sample-frac', '0.1', '--out', 'reports/criteo'],
    capture_output=True, text=True,
)
print(result.stdout[-3000:])
print(result.stderr[-3000:])
if result.returncode != 0:
    raise RuntimeError('The study failed; the error is printed above.')

In [ ]:
from IPython.display import Image, Markdown, display

# Colab's Markdown output cannot load local image files, so show the text and the
# figures separately.
report = open('reports/criteo/REPORT.md').read()
display(Markdown('\n'.join(l for l in report.splitlines() if not l.startswith('!['))))
for name in ('criteo_qini', 'criteo_deciles'):
    display(Image(f'reports/criteo/figures/{name}.png', width=700))

The same analysis is available from Python for further exploration:

In [ ]:
from uplift_lab.data import load_criteo
from uplift_lab.experiment import difference_in_means, sample_ratio_mismatch

df = load_criteo('data/criteo-research-uplift-v2.1.csv.gz', sample_frac=0.1)
print(sample_ratio_mismatch(df['treatment'], 0.85))
print(difference_in_means(df['visit'], df['treatment']))